In [ ]:
!pip install -q datasets scikit-learn nltk


In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

output_dir = "/content/drive/MyDrive/TFIDF_SVM_MirageNews"
os.makedirs(output_dir, exist_ok=True)

print("Output dir:", output_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/TFIDF_SVM_MirageNews


In [ ]:
from datasets import load_dataset

dataset = load_dataset("anson-huang/mirage-news")
dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

data/test1_nyt_mj-00000-of-00001.parquet:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

...

Generating test2_bbc_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test3_cnn_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test4_bbc_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test5_cnn_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})

In [ ]:
X_train = dataset["train"]["text"]
y_train = dataset["train"]["label"]

X_val = dataset["validation"]["text"]
y_val = dataset["validation"]["label"]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print("Train shape:", X_train_tfidf.shape)
print("Val shape:", X_val_tfidf.shape)


Train shape: (10000, 5000)
Val shape: (2500, 5000)


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

svm.fit(X_train_tfidf, y_train)


LinearSVC(class_weight='balanced', max_iter=10000)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

y_pred = svm.predict(X_val_tfidf)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_val, y_pred, average="binary"
)

acc = accuracy_score(y_val, y_pred)

# LinearSVC không có predict_proba → dùng decision_function
scores = svm.decision_function(X_val_tfidf)
auc = roc_auc_score(y_val, scores)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "auc": auc
})


{'accuracy': 0.8108, 'precision': 0.8522212148685403, 'recall': 0.752, 'f1': 0.7989800254993625, 'auc': np.float64(0.87853536)}


In [ ]:
import joblib
import os

output_dir = "/content/drive/MyDrive/TFIDF_SVM_MirageNews"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(tfidf, f"{output_dir}/tfidf.pkl")
joblib.dump(svm, f"{output_dir}/svm.pkl")

print("Saved to:", output_dir)


Saved to: /content/drive/MyDrive/TFIDF_SVM_MirageNews
